## Import thư viện

In [20]:
from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve,
)
import joblib
warnings.filterwarnings("ignore")

## Đường dẫn dữ liệu

In [21]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    THU_MUC_DU_AN = Path("/content/drive/MyDrive/ADHD OneStep/training model")
except Exception:
    THU_MUC_DU_AN = Path(".")

TEP_DU_LIEU = THU_MUC_DU_AN / "train_data.jsonl"
print("Tệp dữ liệu:", TEP_DU_LIEU)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tệp dữ liệu: /content/drive/MyDrive/ADHD OneStep/training model/sample_events_v4_1_large.jsonl


## Đọc dữ liệu

In [22]:
su_kien = []
with TEP_DU_LIEU.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        e = json.loads(line)
        e["timestamp"] = pd.Timestamp(e["timestamp"])
        e["event_type"] = str(e["event_type"]).upper().strip()
        su_kien.append(e)

su_kien = sorted(
    su_kien,
    key=lambda x: (x.get("user_id",""), x["timestamp"], x.get("event_id",0))
)

print("Tổng số sự kiện:", len(su_kien))
print("Số người dùng:", len({e.get("user_id") for e in su_kien}))

Tổng số sự kiện: 16693
Số người dùng: 40


## Quy tắc nhãn recall

In [23]:
CHI_DUNG_GOI_Y_CHUNG_CHO_RECALL = True

CAC_LOAI_GOI_Y_CHUNG = {
    "GENERIC", "GENERIC_CUE", "CUE", "LEVEL1", "PROMPT_LEVEL_1"
}
CAC_LOAI_NHAC_TRUC_TIEP = {
    "EXPLICIT", "DIRECT", "TASK_NAME", "FULL_REMINDER"
}

def la_goi_y_chung(decision):
    data = decision.get("data", {}) or {}
    prompt_level = data.get("prompt_level")
    reminder_type = str(data.get("reminder_type", "")).upper().strip()

    if reminder_type in CAC_LOAI_NHAC_TRUC_TIEP:
        return False
    if prompt_level == 1:
        return True
    return reminder_type in CAC_LOAI_GOI_Y_CHUNG

## Hàm tiện ích

In [24]:
def tinh_ty_le_an_toan(tu_so, mau_so):
    return tu_so / mau_so if mau_so else np.nan

def xac_dinh_buoi_trong_ngay(hour):
    if 5 <= hour < 12: return "buổi_sáng"
    if 12 <= hour < 17: return "buổi_chiều"
    if 17 <= hour < 22: return "buổi_tối"
    return "ban_đêm"

def lay_tap_lan_thuc_hien(history, event_type_set):
    return {
        e.get("occurrence_id")
        for e in history
        if e.get("event_type") in event_type_set
        and e.get("occurrence_id") is not None
    }

def lay_su_kien_tao_nhiem_vu_truoc(su_kien, occurrence_id, decision_time):
    xs = [
        e for e in su_kien
        if e.get("occurrence_id") == occurrence_id
        and e.get("event_type") == "TASK_CREATED"
        and e["timestamp"] < decision_time
    ]
    xs.sort(key=lambda x: x["timestamp"])
    return xs[-1] if xs else None

def lay_su_kien_tuong_lai_dau_tien(su_kien, occurrence_id, event_types, start, end):
    xs = [
        e for e in su_kien
        if e.get("occurrence_id") == occurrence_id
        and e.get("event_type") in event_types
        and start < e["timestamp"] <= end
    ]
    xs.sort(key=lambda x: x["timestamp"])
    return xs[0] if xs else None

def lay_lich_su_trong_khoang(history, t, so_ngay):
    moc = t - pd.Timedelta(days=so_ngay)
    return [e for e in history if e["timestamp"] >= moc]

def tinh_ty_le_recall(history):
    thanh_cong = lay_tap_lan_thuc_hien(history, {"RECALL_SUCCESS","USER_REMEMBERED"})
    that_bai = lay_tap_lan_thuc_hien(history, {"RECALL_FAILURE","USER_FORGOT"})
    tong = thanh_cong | that_bai
    return tinh_ty_le_an_toan(len(thanh_cong), len(tong))

def tinh_chuoi_recall_gan_nhat(history):
    recall_events = [
        e for e in history
        if e.get("event_type") in {
            "RECALL_SUCCESS","USER_REMEMBERED",
            "RECALL_FAILURE","USER_FORGOT"
        }
    ]
    recall_events.sort(key=lambda x: x["timestamp"])
    if not recall_events:
        return 0, 0

    cuoi_la_thanh_cong = recall_events[-1]["event_type"] in {"RECALL_SUCCESS","USER_REMEMBERED"}
    do_dai = 0
    for e in reversed(recall_events):
        ok = e["event_type"] in {"RECALL_SUCCESS","USER_REMEMBERED"}
        if ok != cuoi_la_thanh_cong:
            break
        do_dai += 1
    return (do_dai,0) if cuoi_la_thanh_cong else (0,do_dai)

## Đặc trưng lịch sử

In [25]:
def tinh_dac_trung_lich_su(history, t, so_ngay):
    hist = lay_lich_su_trong_khoang(history, t, so_ngay)

    reminded = lay_tap_lan_thuc_hien(hist, {"REMINDER_SENT"})
    completed = lay_tap_lan_thuc_hien(hist, {"TASK_COMPLETED"})
    snoozed = lay_tap_lan_thuc_hien(hist, {"SNOOZE"})
    forgot = lay_tap_lan_thuc_hien(hist, {"USER_FORGOT","RECALL_FAILURE"})

    reminder_times, start_times = {}, {}
    for e in hist:
        oid = e.get("occurrence_id")
        if oid is None:
            continue
        if e["event_type"] == "REMINDER_SENT":
            reminder_times[oid] = e["timestamp"]
        elif e["event_type"] == "TASK_STARTED":
            start_times[oid] = e["timestamp"]

    delays = []
    for oid, rt in reminder_times.items():
        st = start_times.get(oid)
        if st is not None and st >= rt:
            delays.append((st-rt).total_seconds()/60)

    return {
        f"completion_rate_{so_ngay}d": tinh_ty_le_an_toan(len(completed & reminded), len(reminded)),
        f"snooze_rate_{so_ngay}d": tinh_ty_le_an_toan(len(snoozed & reminded), len(reminded)),
        f"forget_rate_{so_ngay}d": tinh_ty_le_an_toan(len(forgot & reminded), len(reminded)),
        f"recall_rate_{so_ngay}d": tinh_ty_le_recall(hist),
        f"avg_start_delay_{so_ngay}d": float(np.mean(delays)) if delays else np.nan,
        f"reminder_count_{so_ngay}d": len(reminded),
    }

## Feature feedback người dùng

In [26]:
def tinh_dac_trung_feedback(history, t, so_ngay=7):
    hist = lay_lich_su_trong_khoang(history, t, so_ngay)
    feedbacks = [e for e in hist if e.get("event_type") == "REMINDER_FEEDBACK"]

    helpful = not_helpful = too_strong = not_needed = 0
    for e in feedbacks:
        data = e.get("data", {}) or {}
        if data.get("helpful") is True: helpful += 1
        elif data.get("helpful") is False: not_helpful += 1
        if str(data.get("strength","")).upper() == "TOO_STRONG": too_strong += 1
        if bool(data.get("not_needed", False)): not_needed += 1

    total = len(feedbacks)
    return {
        "feedback_count_7d": total,
        "helpful_rate_7d": tinh_ty_le_an_toan(helpful,total),
        "not_helpful_rate_7d": tinh_ty_le_an_toan(not_helpful,total),
        "too_strong_rate_7d": tinh_ty_le_an_toan(too_strong,total),
        "not_needed_rate_7d": tinh_ty_le_an_toan(not_needed,total),
    }

## Xây dựng feature

In [27]:
def xay_dung_dac_trung_v4(su_kien, decision):
    t = decision["timestamp"]
    user_id = decision.get("user_id")
    task_id = decision.get("task_id")
    occ = decision.get("occurrence_id")

    history = [
        e for e in su_kien
        if e.get("user_id") == user_id and e["timestamp"] < t
    ]

    created = lay_su_kien_tao_nhiem_vu_truoc(su_kien, occ, t)
    task_data = (created or {}).get("data", {}) or {}
    reminder_data = decision.get("data", {}) or {}

    same_task_hist = [e for e in history if e.get("task_id") == task_id]
    same_task_reminded = lay_tap_lan_thuc_hien(same_task_hist, {"REMINDER_SENT"})
    same_task_completed = lay_tap_lan_thuc_hien(same_task_hist, {"TASK_COMPLETED"})

    recall_ok_same = lay_tap_lan_thuc_hien(same_task_hist, {"RECALL_SUCCESS","USER_REMEMBERED"})
    recall_fail_same = lay_tap_lan_thuc_hien(same_task_hist, {"RECALL_FAILURE","USER_FORGOT"})
    recall_all_same = recall_ok_same | recall_fail_same

    success_streak, fail_streak = tinh_chuoi_recall_gan_nhat(history)
    first_hist_time = min((e["timestamp"] for e in history), default=t)

    hour, weekday = t.hour, t.weekday()

    row = {
        "user_id": user_id,
        "task_id": task_id,
        "occurrence_id": occ,
        "decision_time": t,
        "hour": hour,
        "weekday": weekday,
        "is_weekend": int(weekday >= 5),
        "time_of_day": xac_dinh_buoi_trong_ngay(hour),
        "hour_sin": math.sin(2*math.pi*hour/24),
        "hour_cos": math.cos(2*math.pi*hour/24),
        "task_type": task_data.get("task_type","UNKNOWN"),
        "priority": task_data.get("priority"),
        "estimated_minutes": task_data.get("estimated_minutes"),
        "prompt_level": reminder_data.get("prompt_level"),
        "reminder_type": reminder_data.get("reminder_type","UNKNOWN"),
        "channel": reminder_data.get("channel","UNKNOWN"),
        "same_task_completion_rate": tinh_ty_le_an_toan(
            len(same_task_completed & same_task_reminded), len(same_task_reminded)
        ),
        "same_task_recall_rate": tinh_ty_le_an_toan(
            len(recall_ok_same), len(recall_all_same)
        ),
        "recall_success_streak": success_streak,
        "recall_failure_streak": fail_streak,
        "event_count_before": len(history),
        "history_days": max(0.0,(t-first_hist_time).total_seconds()/86400),
        "is_cold_start": int(len(history) < 10),
    }

    for so_ngay in (1,3,7):
        row.update(tinh_dac_trung_lich_su(history,t,so_ngay))

    row.update(tinh_dac_trung_feedback(history,t,7))
    return row

## Xây dựng nhãn

In [28]:
def xay_dung_nhan_v4(su_kien, decision):
    t = decision["timestamp"]
    occ = decision.get("occurrence_id")

    out = {
        "y_recall_success": np.nan,
        "y_started_within_10m": 0,
        "y_complete_within_30m": 0,
        "y_complete_within_60m": 0,
        "y_snooze": 0,
        "y_forgot": np.nan,
        "recall_label_eligible": int(la_goi_y_chung(decision)),
    }

    duoc_gan_nhan_recall = la_goi_y_chung(decision) if CHI_DUNG_GOI_Y_CHUNG_CHO_RECALL else True

    if duoc_gan_nhan_recall:
        recall = lay_su_kien_tuong_lai_dau_tien(
            su_kien, occ, {"RECALL_SUCCESS","RECALL_FAILURE"},
            t, t + pd.Timedelta(minutes=2)
        )
        if recall is not None:
            out["y_recall_success"] = int(recall["event_type"] == "RECALL_SUCCESS")

    started = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ, {"TASK_STARTED"},
        t, t + pd.Timedelta(minutes=60)
    )
    if started is not None:
        delay = (started["timestamp"]-t).total_seconds()/60
        out["y_started_within_10m"] = int(delay <= 10)

    completed = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ, {"TASK_COMPLETED"},
        t, t + pd.Timedelta(minutes=180)
    )
    if completed is not None:
        delay = (completed["timestamp"]-t).total_seconds()/60
        out["y_complete_within_30m"] = int(delay <= 30)
        out["y_complete_within_60m"] = int(delay <= 60)

    snooze = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ, {"SNOOZE"},
        t, t + pd.Timedelta(minutes=10)
    )
    out["y_snooze"] = int(snooze is not None)

    forgot = lay_su_kien_tuong_lai_dau_tien(
        su_kien, occ,
        {"USER_FORGOT","USER_REMEMBERED","RECALL_FAILURE","RECALL_SUCCESS"},
        t, t + pd.Timedelta(minutes=10)
    )
    if forgot is not None:
        out["y_forgot"] = int(
            forgot["event_type"] in {"USER_FORGOT","RECALL_FAILURE"}
        )

    return out

## Tạo dataset

In [29]:
cac_diem_quyet_dinh = [e for e in su_kien if e.get("event_type") == "REMINDER_SENT"]
cac_ban_ghi_hoc = []

for d in cac_diem_quyet_dinh:
    row = {}
    row.update(xay_dung_dac_trung_v4(su_kien,d))
    row.update(xay_dung_nhan_v4(su_kien,d))
    cac_ban_ghi_hoc.append(row)

df = pd.DataFrame(cac_ban_ghi_hoc)

print("Số điểm ra quyết định:", len(df))
print("Số mẫu recall đủ điều kiện:", int(df["y_recall_success"].notna().sum()))
display(df.head())

Số điểm ra quyết định: 4091
Số mẫu recall đủ điều kiện: 2251


,user_id,task_id,occurrence_id,decision_time,hour,weekday,is_weekend,time_of_day,hour_sin,hour_cos,...,not_helpful_rate_7d,too_strong_rate_7d,not_needed_rate_7d,y_recall_success,y_started_within_10m,y_complete_within_30m,y_complete_within_60m,y_snooze,y_forgot,recall_label_eligible
0,U001,EXERCISE,EXERCISE_U001_2026-08-01_1,2026-08-01 18:38:00+07:00,18,5,1,buổi_tối,-1.000000,-1.836970e-16,...,NaN,NaN,NaN,NaN,1,0,0,0,NaN,0
1,U001,BAG,BAG_U001_2026-08-02_76,2026-08-02 07:04:00+07:00,7,6,1,buổi_sáng,0.965926,-2.588190e-01,...,NaN,NaN,NaN,NaN,1,1,1,0,NaN,0
2,U001,WATER,WATER_U001_2026-08-02_75,2026-08-02 10:42:00+07:00,10,6,1,buổi_sáng,0.500000,-8.660254e-01,...,NaN,NaN,NaN,1.0,1,1,1,0,0.0,1
3,U001,EXERCISE,EXERCISE_U001_2026-08-02_77,2026-08-02 20:15:00+07:00,20,6,1,buổi_tối,-0.866025,5.000000e-01,...,0.0,0.0,0.0,0.0,1,0,0,0,1.0,1
4,U001,BAG,BAG_U001_2026-08-03_139,2026-08-03 07:15:00+07:00,7,0,0,buổi_sáng,0.965926,-2.588190e-01,...,0.0,0.0,0.0,1.0,1,0,0,0,0.0,1


## Audit nhãn recall

In [30]:
def tao_bang_audit_recall(su_kien, cac_diem_quyet_dinh, so_mau=30):
    rows = []
    for d in cac_diem_quyet_dinh:
        t = d["timestamp"]
        occ = d.get("occurrence_id")
        future = [
            e for e in su_kien
            if e.get("occurrence_id") == occ
            and t < e["timestamp"] <= t + pd.Timedelta(minutes=10)
        ]
        future.sort(key=lambda x:x["timestamp"])

        rows.append({
            "user_id": d.get("user_id"),
            "occurrence_id": occ,
            "decision_time": t,
            "prompt_level": (d.get("data",{}) or {}).get("prompt_level"),
            "reminder_type": (d.get("data",{}) or {}).get("reminder_type"),
            "la_goi_y_chung": la_goi_y_chung(d),
            "future_events_10m": " → ".join(e["event_type"] for e in future)
        })

    audit = pd.DataFrame(rows)
    if len(audit) <= so_mau:
        return audit
    return audit.sample(so_mau, random_state=42).sort_values("decision_time")

display(tao_bang_audit_recall(su_kien,cac_diem_quyet_dinh,30))

,user_id,occurrence_id,decision_time,prompt_level,reminder_type,la_goi_y_chung,future_events_10m
3567,U036,MATH_U036_2026-08-03_191,2026-08-03 21:00:00+07:00,1,generic,True,RECALL_SUCCESS → SNOOZE → TASK_STARTED
1027,U011,WATER_U011_2026-08-06_356,2026-08-06 13:06:00+07:00,2,explicit,False,
109,U002,MATH_U002_2026-08-06_341,2026-08-06 19:57:00+07:00,0,delayed_check,False,
325,U004,ENGLISH_U004_2026-08-07_407,2026-08-07 14:38:00+07:00,0,delayed_check,False,REMINDER_FEEDBACK
3580,U036,PROJECT_U036_2026-08-09_588,2026-08-09 15:44:00+07:00,0,delayed_check,False,TASK_STARTED
1034,U011,REVISION_U011_2026-08-09_545,2026-08-09 21:51:00+07:00,1,generic,True,RECALL_SUCCESS
1330,U014,READ_U014_2026-08-13_826,2026-08-13 18:46:00+07:00,1,generic,True,RECALL_SUCCESS → TASK_STARTED
2775,U028,REVISION_U028_2026-08-14_904,2026-08-14 14:51:00+07:00,1,generic,True,RECALL_SUCCESS
1641,U017,PROJECT_U017_2026-08-18_1133,2026-08-18 15:19:00+07:00,0,delayed_check,False,SNOOZE
2787,U028,READ_U028_2026-08-20_1280,2026-08-20 18:10:00+07:00,1,generic,True,SNOOZE → RECALL_SUCCESS


## Chuẩn bị pipeline

In [31]:
CAC_COT_BO_QUA = {"user_id","occurrence_id","decision_time"}
TAT_CA_NHAN = {
    "y_recall_success","y_started_within_10m",
    "y_complete_within_30m","y_complete_within_60m",
    "y_snooze","y_forgot","recall_label_eligible"
}

def lay_cot_dac_trung(work):
    return [c for c in work.columns if c not in CAC_COT_BO_QUA and c not in TAT_CA_NHAN]

def tao_tien_xu_ly(X_train, feature_cols):
    cat_cols = [c for c in feature_cols if X_train[c].dtype == "object"]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols)
    ])

## Các model so sánh

In [32]:
def tao_cac_mo_hinh():
    return {
        "Dummy": DummyClassifier(strategy="prior", random_state=42),
        "Logistic Regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=400, max_depth=8, min_samples_leaf=3,
            class_weight="balanced", random_state=42, n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=150, learning_rate=0.04, max_depth=2, random_state=42
        ),
    }

## Metrics

In [33]:
def tinh_specificity(y_true,y_pred):
    cm = confusion_matrix(y_true,y_pred,labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    return tn/(tn+fp) if (tn+fp) else np.nan

def tinh_cac_chi_so(y_true,y_pred,y_prob):
    result = {
        "accuracy": accuracy_score(y_true,y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true,y_pred),
        "precision": precision_score(y_true,y_pred,zero_division=0),
        "recall": recall_score(y_true,y_pred,zero_division=0),
        "specificity": tinh_specificity(y_true,y_pred),
        "f1": f1_score(y_true,y_pred,zero_division=0),
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "brier": np.nan,
    }
    if y_prob is not None and len(np.unique(y_true)) > 1:
        result["roc_auc"] = roc_auc_score(y_true,y_prob)
        result["pr_auc"] = average_precision_score(y_true,y_prob)
        result["brier"] = brier_score_loss(y_true,y_prob)
    return result

def lay_xac_suat_lop_1(pipe,X):
    prob = pipe.predict_proba(X)
    classes = list(pipe.named_steps["model"].classes_)
    return prob[:,classes.index(1)] if 1 in classes else None

## Walk-forward validation

In [34]:
def danh_gia_walk_forward(df,target,n_splits=5):
    work = df[df[target].notna()].sort_values("decision_time").reset_index(drop=True)
    feature_cols = lay_cot_dac_trung(work)
    X = work[feature_cols]
    y = work[target].astype(int)

    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []

    for fold,(train_idx,test_idx) in enumerate(tscv.split(X),start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

        for ten_mo_hinh,model in tao_cac_mo_hinh().items():
            pipe = Pipeline([
                ("preprocess", tao_tien_xu_ly(X_train,feature_cols)),
                ("model", clone(model))
            ])
            pipe.fit(X_train,y_train)
            y_pred = pipe.predict(X_test)
            y_prob = lay_xac_suat_lop_1(pipe,X_test)
            metrics = tinh_cac_chi_so(y_test,y_pred,y_prob)

            rows.append({
                "target":target,
                "fold":fold,
                "model":ten_mo_hinh,
                "train_rows":len(train_idx),
                "test_rows":len(test_idx),
                "positive_rate_test":float(y_test.mean()),
                **metrics
            })

    ket_qua = pd.DataFrame(rows)
    tong_hop = (
        ket_qua.groupby("model",as_index=False)
        .agg(
            roc_auc_mean=("roc_auc","mean"),
            roc_auc_std=("roc_auc","std"),
            pr_auc_mean=("pr_auc","mean"),
            recall_mean=("recall","mean"),
            specificity_mean=("specificity","mean"),
            f1_mean=("f1","mean"),
            balanced_accuracy_mean=("balanced_accuracy","mean"),
            brier_mean=("brier","mean"),
        )
        .sort_values("roc_auc_mean",ascending=False)
    )
    return {"work":work,"feature_columns":feature_cols,"fold_results":ket_qua,"summary":tong_hop}

## Chạy đánh giá recall và completion

In [35]:
cv_tu_nho = danh_gia_walk_forward(df,"y_recall_success",5)
cv_hoan_thanh = danh_gia_walk_forward(df,"y_complete_within_30m",5)

print("Recall model comparison:")
display(cv_tu_nho["summary"])

print("Completion model comparison:")
display(cv_hoan_thanh["summary"])

Recall model comparison:


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,recall_mean,specificity_mean,f1_mean,balanced_accuracy_mean,brier_mean
3,Random Forest,0.757310,0.035034,0.757111,0.704217,0.700421,0.699270,0.702319,0.204069
2,Logistic Regression,0.751974,0.028604,0.756092,0.735758,0.596298,0.682794,0.666028,0.219502
1,Gradient Boosting,0.750631,0.026715,0.741108,0.739861,0.619562,0.694321,0.679711,0.206061
0,Dummy,0.500000,0.000000,0.492800,0.600000,0.400000,0.390034,0.500000,0.250483


Completion model comparison:


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,recall_mean,specificity_mean,f1_mean,balanced_accuracy_mean,brier_mean
3,Random Forest,0.881383,0.011576,0.494329,0.532031,0.929373,0.501402,0.730702,0.090569
1,Gradient Boosting,0.873231,0.020502,0.478573,0.236538,0.975593,0.301744,0.606065,0.077942
2,Logistic Regression,0.866690,0.019620,0.486249,0.778064,0.767277,0.427323,0.772671,0.147196
0,Dummy,0.500000,0.000000,0.111013,0.000000,1.000000,0.000000,0.500000,0.098779


## Chọn model tốt nhất và train final 70/30

In [36]:
def chon_model_tot_nhat(cv_result):
    bang = cv_result["summary"].dropna(subset=["roc_auc_mean"])
    bang = bang[bang["model"] != "Dummy"]
    return "Random Forest" if len(bang)==0 else bang.iloc[0]["model"]

def huan_luyen_final(df,target,ten_model,train_ratio=0.7):
    work = df[df[target].notna()].sort_values("decision_time").reset_index(drop=True)
    feature_cols = lay_cot_dac_trung(work)
    cut = int(len(work)*train_ratio)
    cut = min(max(cut,1),len(work)-1)

    train,test = work.iloc[:cut], work.iloc[cut:]
    X_train,y_train = train[feature_cols], train[target].astype(int)
    X_test,y_test = test[feature_cols], test[target].astype(int)

    pipe = Pipeline([
        ("preprocess",tao_tien_xu_ly(X_train,feature_cols)),
        ("model",clone(tao_cac_mo_hinh()[ten_model]))
    ])
    pipe.fit(X_train,y_train)

    y_pred = pipe.predict(X_test)
    y_prob = lay_xac_suat_lop_1(pipe,X_test)

    return {
        "version":"PCBM-v4",
        "target":target,
        "model_name":ten_model,
        "pipeline":pipe,
        "feature_columns":feature_cols,
        "train_rows":len(train),
        "test_rows":len(test),
        "y_test":y_test,
        "y_pred":y_pred,
        "y_prob":y_prob,
        "metrics":tinh_cac_chi_so(y_test,y_pred,y_prob),
    }

ten_model_tu_nho = chon_model_tot_nhat(cv_tu_nho)
ten_model_hoan_thanh = chon_model_tot_nhat(cv_hoan_thanh)

mo_hinh_tu_nho_v4 = huan_luyen_final(df,"y_recall_success",ten_model_tu_nho)
mo_hinh_hoan_thanh_v4 = huan_luyen_final(df,"y_complete_within_30m",ten_model_hoan_thanh)

display(pd.DataFrame([
    {"Bài toán":"Tự nhớ","Model":mo_hinh_tu_nho_v4["model_name"],**mo_hinh_tu_nho_v4["metrics"]},
    {"Bài toán":"Hoàn thành ≤30m","Model":mo_hinh_hoan_thanh_v4["model_name"],**mo_hinh_hoan_thanh_v4["metrics"]},
]))

,Bài toán,Model,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,brier
0,Tự nhớ,Random Forest,0.698225,0.696866,0.695793,0.661538,0.732194,0.678233,0.768810,0.758968,0.199534
1,Hoàn thành ≤30m,Random Forest,0.866450,0.755267,0.417526,0.613636,0.896898,0.496933,0.870251,0.398151,0.099895


## 18. Điều kiện an toàn khi đề xuất giảm mức hỗ trợ

In [37]:
NGUONG_AUC_TOI_THIEU_DE_XET_GIAM = 0.70
SO_SU_KIEN_LICH_SU_TOI_THIEU = 20
CHUOI_RECALL_THANH_CONG_TOI_THIEU = 3

def de_xuat_muc_ho_tro_v4(
    p_recall,
    muc_hien_tai,
    event_count_before,
    recall_success_streak,
    model_roc_auc,
):
    du_tin_cay = (
        model_roc_auc >= NGUONG_AUC_TOI_THIEU_DE_XET_GIAM
        and event_count_before >= SO_SU_KIEN_LICH_SU_TOI_THIEU
        and recall_success_streak >= CHUOI_RECALL_THANH_CONG_TOI_THIEU
    )

    if p_recall >= 0.80 and du_tin_cay and muc_hien_tai > 0:
        return {
            "muc_de_xuat":muc_hien_tai-1,
            "hanh_dong":"DE_XUAT_GIAM",
            "tu_dong_ap_dung":False
        }

    if p_recall < 0.45 and muc_hien_tai < 2:
        return {
            "muc_de_xuat":muc_hien_tai+1,
            "hanh_dong":"DE_XUAT_TANG",
            "tu_dong_ap_dung":False
        }

    return {
        "muc_de_xuat":muc_hien_tai,
        "hanh_dong":"GIU",
        "tu_dong_ap_dung":False
    }

## 19. Lưu model

In [38]:
THU_MUC_MODEL = Path("models_v4")
THU_MUC_MODEL.mkdir(parents=True,exist_ok=True)

joblib.dump(mo_hinh_tu_nho_v4, THU_MUC_MODEL/"pcbm_recall_v4.joblib")
joblib.dump(mo_hinh_hoan_thanh_v4, THU_MUC_MODEL/"pcbm_completion_30m_v4.joblib")

print("Đã lưu model vào:", THU_MUC_MODEL.resolve())

Đã lưu model vào: /content/models_v4
